In [10]:
import cv2
import numpy as np
import time
import mediapipe as mp
mp_drawing = mp.solutions.drawing_utils
mp_pose = mp.solutions.pose
draw_style=mp_drawing.DrawingSpec(thickness=1, circle_radius=1, color=(0,0,255))
draw_connection=mp_drawing.DrawingSpec(thickness=1, circle_radius=1, color=(0,255,0))

In [3]:
pose_landmarks = {
    0: "NOSE",
    1: "LEFT_EYE_INNER",
    2: "LEFT_EYE",
    3: "LEFT_EYE_OUTER",
    4: "RIGHT_EYE_INNER",
    5: "RIGHT_EYE",
    6: "RIGHT_EYE_OUTER",
    7: "LEFT_EAR",
    8: "RIGHT_EAR",
    9: "MOUTH_LEFT",
    10: "MOUTH_RIGHT",
    11: "LEFT_SHOULDER",
    12: "RIGHT_SHOULDER",
    13: "LEFT_ELBOW",
    14: "RIGHT_ELBOW",
    15: "LEFT_WRIST",
    16: "RIGHT_WRIST",
    17: "LEFT_PINKY",
    18: "RIGHT_PINKY",
    19: "LEFT_INDEX",
    20: "RIGHT_INDEX",
    21: "LEFT_THUMB",
    22: "RIGHT_THUMB",
    23: "LEFT_HIP",
    24: "RIGHT_HIP",
    25: "LEFT_KNEE",
    26: "RIGHT_KNEE",
    27: "LEFT_ANKLE",
    28: "RIGHT_ANKLE",
    29: "LEFT_HEEL",
    30: "RIGHT_HEEL",
    31: "LEFT_FOOT_INDEX",
    32: "RIGHT_FOOT_INDEX"
}

In [ ]:
cap=cv2.VideoCapture(1)
cap.set(cv2.CAP_PROP_FRAME_WIDTH, 1280)
cap.set(cv2.CAP_PROP_FRAME_HEIGHT, 720)

prev_frame_time = time.time()
fps = 0
frame_counter = 0

with mp_pose.Pose(min_detection_confidence=0.5, min_tracking_confidence=0.5) as pose:
    while cap.isOpened():
        ret, frame = cap.read()
        if not ret:
            break
        frame_counter+=1
       
        # Recolor image to RGB
        frame=cv2.rotate(frame, cv2.ROTATE_90_CLOCKWISE)
        image = cv2.cvtColor(frame, cv2.COLOR_BGR2RGB)
        image.flags.writeable = False
        results = pose.process(image)
        image.flags.writeable = True
        image = cv2.cvtColor(image, cv2.COLOR_RGB2BGR)
        if results.pose_landmarks:
            landmarks = results.pose_landmarks.landmark

            mp_drawing.draw_landmarks(image, results.pose_landmarks, mp_pose.  POSE_CONNECTIONS, draw_style, draw_connection)
        
            h, w = image.shape[:2]
        
            x = [int(lm.x * w) for lm in results.pose_landmarks.landmark]
            y = [int(lm.y * h) for lm in results.pose_landmarks.landmark]

            x1, y1 = max(0, min(x)-20), max(0, min(y)-20)
            x2, y2 = min(w, max(x)+20), min(h, max(y)+20)

            cv2.rectangle(image, (x1, y1), (x2, y2), (255, 0, 0), 2)
            cv2.putText(image, "Human Detection", (x1, y1 - 10), cv2.FONT_HERSHEY_SIMPLEX, 0.9, (255, 0, 0), 2)

         # Stable FPS
        new_frame_time = time.time()
        current_fps = 1 / (new_frame_time - prev_frame_time)
        prev_frame_time = new_frame_time
        fps = 0.9 * fps + 0.1 * current_fps
        
        cv2.putText(frame, "FPS: "+str(int(fps)), (7, 70), cv2.FONT_HERSHEY_SIMPLEX, 3, (100, 255, 0), 3, cv2.LINE_AA)
        cv2.putText(frame, "Frame: "+str(frame_counter), (7, 120), cv2.FONT_HERSHEY_SIMPLEX, 3, (100, 255, 0), 3, cv2.LINE_AA)

        cv2.imshow('Mediaipe Feed', image)
        key=cv2.waitKey(1)
        if key==ord('q'):
            break
    cap.release()
    cv2.destroyAllWindows()


In [50]:
landmarks

[x: 0.746564746
y: 0.304209173
z: -0.146508291
visibility: 0.988505602
, x: 0.74894309
y: 0.304962575
z: -0.157193705
visibility: 0.987751603
, x: 0.74909848
y: 0.304955035
z: -0.157210395
visibility: 0.985197902
, x: 0.749087274
y: 0.304838061
z: -0.157207415
visibility: 0.985881
, x: 0.748918176
y: 0.306265473
z: -0.15994437
visibility: 0.992558837
, x: 0.748858869
y: 0.306932598
z: -0.160032779
visibility: 0.992623687
, x: 0.748595536
y: 0.307293236
z: -0.160215825
visibility: 0.993281424
, x: 0.741069555
y: 0.30578509
z: -0.164024726
visibility: 0.987835705
, x: 0.745519161
y: 0.311388433
z: -0.176221758
visibility: 0.993028104
, x: 0.734401643
y: 0.311798811
z: -0.142881274
visibility: 0.985948622
, x: 0.742525399
y: 0.310319543
z: -0.146848291
visibility: 0.991072655
, x: 0.705626905
y: 0.315636158
z: -0.132702559
visibility: 0.996550202
, x: 0.711511493
y: 0.328151166
z: -0.1591333
visibility: 0.998573363
, x: 0.715801299
y: 0.30131
z: -0.0879273489
visibility: 0.213770524
, x: 

In [51]:
for lm in mp_pose.PoseLandmark:
    print(lm)

0
1
2
3
4
5
6
7
8
9
10
11
12
13
14
15
16
17
18
19
20
21
22
23
24
25
26
27
28
29
30
31
32


In [52]:
landmarks[mp_pose.PoseLandmark.LEFT_SHOULDER.value]

x: 0.705626905
y: 0.315636158
z: -0.132702559
visibility: 0.996550202

In [53]:
landmarks[mp_pose.PoseLandmark.LEFT_WRIST.value]

x: 0.726312578
y: 0.289242923
z: -0.0899951905
visibility: 0.170318082

In [11]:
def calculate_angle(a,b,c):
  a=np.array(a) # first
  b=np.array(b) # mid
  c=np.array(c) # end

  radians=np.arctan2(c[1]-b[1], c[0]-b[0])-np.arctan2(a[1]-b[1], a[0]-b[0])
  angle=np.abs(radians*180.0/np.pi)
  if angle>180.0:
    angle=360-angle
  return angle


In [58]:


shoulder = [
    landmarks[mp_pose.PoseLandmark.LEFT_SHOULDER.value].x,
    landmarks[mp_pose.PoseLandmark.LEFT_SHOULDER.value].y
]

elbow = [
    landmarks[mp_pose.PoseLandmark.LEFT_ELBOW.value].x,
    landmarks[mp_pose.PoseLandmark.LEFT_ELBOW.value].y
]

wrist = [
    landmarks[mp_pose.PoseLandmark.LEFT_WRIST.value].x,
    landmarks[mp_pose.PoseLandmark.LEFT_WRIST.value].y
]

# MediaPipe Pose Landmark Extraction and Elbow Angle Calculation


In [21]:
cap=cv2.VideoCapture(1)
h,w=1082,1980
cap.set(cv2.CAP_PROP_FRAME_WIDTH, w)
cap.set(cv2.CAP_PROP_FRAME_HEIGHT, h)

prev_frame_time = time.time()
fps = 0
frame_counter = 0
counter=0
stage=None
with mp_pose.Pose(min_detection_confidence=0.5, min_tracking_confidence=0.5) as pose:
    while cap.isOpened():
        ret, frame = cap.read()
        if not ret:
            break
        frame_counter+=1
       
        # Recolor image to RGB
        # frame=cv2.rotate(frame, cv2.ROTATE_90_CLOCKWISE)
        frame=cv2.flip(frame,-1)
        image = cv2.cvtColor(frame, cv2.COLOR_BGR2RGB)
        image.flags.writeable = False
        results = pose.process(image)
        image.flags.writeable = True
        image = cv2.cvtColor(image, cv2.COLOR_RGB2BGR)
        if results.pose_landmarks:
            landmarks = results.pose_landmarks.landmark

            shoulder=[landmarks[mp_pose.PoseLandmark.LEFT_SHOULDER.value].x,landmarks[mp_pose.PoseLandmark.LEFT_SHOULDER.value].y]

            elbow=[landmarks[mp_pose.PoseLandmark.LEFT_ELBOW.value].x,landmarks[mp_pose.PoseLandmark.LEFT_ELBOW.value].y]

            wrist=[landmarks[mp_pose.PoseLandmark.LEFT_WRIST.value].x,landmarks[mp_pose.PoseLandmark.LEFT_WRIST.value].y]

            angle=calculate_angle(shoulder,elbow,wrist)
            cv2.putText(image, str(angle), tuple(np.multiply(elbow, [w, h]).astype(int)), cv2.FONT_HERSHEY_SIMPLEX, 0.5, (255, 255, 255), 2)

             
            mp_drawing.draw_landmarks(image, results.pose_landmarks, mp_pose.  POSE_CONNECTIONS, draw_style, draw_connection)
            if angle>160:
                stage="down"
            if angle<45 and stage=='down':
                stage="up"
                counter+=1
                print(counter)
           

        
            h, w = image.shape[:2]
        
            x = [int(lm.x * w) for lm in results.pose_landmarks.landmark]
            y = [int(lm.y * h) for lm in results.pose_landmarks.landmark]

            x1, y1 = max(0, min(x)-20), max(0, min(y)-20)
            x2, y2 = min(w, max(x)+20), min(h, max(y)+20)

            cv2.rectangle(image, (x1, y1), (x2, y2), (255, 0, 0), 2)
            cv2.putText(image, "Human Detection", (x1, y1 - 10), cv2.FONT_HERSHEY_SIMPLEX, 0.9, (255, 0, 0), 2)
           

        # Stable FPS
        new_frame_time = time.time()
        current_fps = 1 / (new_frame_time - prev_frame_time)
        prev_frame_time = new_frame_time
        fps = 0.9 * fps + 0.1 * current_fps
        
        cv2.putText(image, "FPS: "+str(int(fps)), (7, 70), cv2.FONT_HERSHEY_SIMPLEX, 1, (100, 255, 0), 2, cv2.LINE_AA)
        cv2.putText(image, "Frame: "+str(frame_counter), (7, 120), cv2.FONT_HERSHEY_SIMPLEX, 1, (100, 255, 0), 2, cv2.LINE_AA)
        cv2.putText(image, 'REPS: '+str(counter), (7,150), cv2.FONT_HERSHEY_SIMPLEX, 1, (100, 255, 0), 2, cv2.LINE_AA)
        cv2.putText(image, 'STAGE: '+str(stage), (7,200), cv2.FONT_HERSHEY_SIMPLEX, 1, (100, 255, 0), 2, cv2.LINE_AA)
       

        cv2.imshow('Mediapipe Feed', image)
        key=cv2.waitKey(1)
        if key==ord('q'):
            break
    cap.release()
    cv2.destroyAllWindows()


1
2
3
4
5
6
7
8
9
10
11
12
13
14
15
16
17
